In [20]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import sys
sys.path.append("../../")

import rateslib as rl
import QuantLib as ql

from Query.IRSwaps.IRSwapQuery import IRSwapQuery 
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapValue import IRSwapValue

In [22]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")

In [23]:
# ts = NYC_tz.localize(datetime.datetime(2026, 4, 10, 17, 00))
ts = "live"
curve = "USD-SOFR-1D-Q12STIRT"

curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle

RLIRSwapCurve(_rl_curve_id='USD-SOFR-1D', _rl_curve_handle=<rl.Curve:USD-SOFR-1D at 0x1b28e8c08d0>, _meta_data={'timestamp': datetime.datetime(2026, 4, 13, 8, 51, 49, 965629, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>), 'id': 'BARCHART_STIRF-RL-USD-SOFR-1D-Q12STIRT-2026-04-13 08:51:49.965629-04:00', 'requested_curve_name': 'USD-SOFR-1D-Q12STIRT', 'curve_name': 'USD-SOFR-1D-Q12STIRT'})

In [24]:
risk_model_tenors = [
	# "fomc_1",
	# "fomc_2",
	# "fomc_3",
	# "fomc_4",
	# "fomc_5",
	# "fomc_6",
	# "fomc_7",
	
	"IMM_1xIMM_2",
    "IMM_2xIMM_3",
    "IMM_3xIMM_4",
    "IMM_4xIMM_5",
    "IMM_5xIMM_6",
    "IMM_6xIMM_7",
    "IMM_7xIMM_8",
    "IMM_8xIMM_9",
    "IMM_9xIMM_10",
    "IMM_10xIMM_11",
    "IMM_11xIMM_12",
    "IMM_12xIMM_13",
    
	# "IMM_13xIMM_14",
    # "IMM_14xIMM_15",
    # "IMM_15xIMM_16",
    # "IMM_16xIMM_17",
	# "IMM_17xIMM_18",
    # "IMM_18xIMM_19",
    # "IMM_19xIMM_20",
    # "IMM_20xIMM_21",
]
rl_risk_instruments = {}
for t in risk_model_tenors:
    outright_query = IRSwapQuery(curve=curve, tenor=t).resolve_query(ts, pricer_or_curve=curve_handle)
    outright_pkg, _ = outright_query.resolve_package(pricer_or_curve=curve_handle)
    rl_risk_instruments[t] = outright_pkg[0]

rl_risk_solver = rl.Solver(
    curves=[curve_handle.handle()],
    instruments=rl_risk_instruments.values(),
    instrument_labels=rl_risk_instruments.keys(),
    s=[r.rate().real for r in rl_risk_instruments.values()],
    id=curve_handle.id(),
    func_tol=1e-8,
    conv_tol=1e-10,
)

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0011s


In [27]:
risk = 100_000

tenor = "IMM_12xIMM_13"
query = IRSwapQuery(curve=curve, tenor=tenor, structure_kwargs={"bpv": risk}).resolve_query(
    ts, pricer_or_curve=curve_handle
)
pkg, rws = query.resolve_package(pricer_or_curve=curve_handle)

vmap = query.build_value_map(pricer_or_curve=curve_handle, package=pkg, risk_weights=rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.NOTIONAL, IRSwapValue.PV01]:
    print(v.name, vmap.apply(value=v))

print(curve_handle.effective_date(pkg[0]))
print(curve_handle.maturity_date(pkg[0]))
# print(vmap.apply(value=IRSwapValue.CARRY_BPS_RUNNING, **{"horizon": "3m"}))
print(vmap.apply(value=IRSwapValue.ROLL_BPS_RUNNING, **{"horizon": "1m"}))

display(rl.Portfolio(pkg).delta(solver=rl_risk_solver).style.format("{:_.0f}"))

RATE 3.5494341897915773
NPV 7.450580596923828e-09
NOTIONAL 4435600921.366873
PV01 100000.00000000001
2029-03-21 00:00:00
2029-06-20 00:00:00
0.5228601319878096


In [18]:
# curve_handle.handle().plot("1y")

queries = [
    # IRSwapQuery(curve=curve, tenor="IMM_H26xIMM_M26", structure_kwargs={"bpv": risk}).resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M26xIMM_U26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U26xIMM_Z26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z26xIMM_H27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H27xIMM_M27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M27xIMM_U27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U27xIMM_Z27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z27xIMM_H28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H28xIMM_M28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M28xIMM_U28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U28xIMM_Z28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z28xIMM_H29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H29xIMM_M29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M29xIMM_U29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U29xIMM_Z29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z29xIMM_H30").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H30xIMM_M30").resolve_query(ts, pricer_or_curve=curve_handle),
]

rl_spots = []
for q in queries:
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    print(f"{q.col_name()}: {pkg[0].rate().real}")


#     rl_spots.append(pkg[0])

# x = [s.__dict__["kwargs"]["termination"] for s in rl_spots]
# y = [s.rate().real for s in rl_spots]

# fig = go.Figure()
# fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="1D"))
# tick_vals = [pd.Timestamp(t).tz_localize(None) for t in x]
# tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]
# fig.update_layout(
#     title=f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]} | spot term curve",
#     template="plotly_dark",
#     margin=dict(l=40, r=20, t=60, b=80),
#     xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
#     height=550,
#     yaxis=dict(showgrid=True),
# )
# fig.update_xaxes(
#     showspikes=True,
#     spikecolor="white",
#     spikesnap="cursor",
#     spikemode="across",
#     showgrid=True,
# )
# fig.update_yaxes(
#     showspikes=True,
#     spikecolor="white",
#     spikesnap="cursor",
#     spikethickness=0.5,
#     showgrid=True,
# )
# fig.show()

USD-SOFR-1D-Q16STIRT IMM_M26xIMM_U26 OUTRIGHT RATE: 3.6985546088551127
USD-SOFR-1D-Q16STIRT IMM_U26xIMM_Z26 OUTRIGHT RATE: 3.7039818441981827
USD-SOFR-1D-Q16STIRT IMM_Z26xIMM_H27 OUTRIGHT RATE: 3.6904076327874544
USD-SOFR-1D-Q16STIRT IMM_H27xIMM_M27 OUTRIGHT RATE: 3.6549245981686136
USD-SOFR-1D-Q16STIRT IMM_M27xIMM_U27 OUTRIGHT RATE: 3.596585655756927
USD-SOFR-1D-Q16STIRT IMM_U27xIMM_Z27 OUTRIGHT RATE: 3.5198076864595684
USD-SOFR-1D-Q16STIRT IMM_Z27xIMM_H28 OUTRIGHT RATE: 3.4729036252787036
USD-SOFR-1D-Q16STIRT IMM_H28xIMM_M28 OUTRIGHT RATE: 3.474214302344052
USD-SOFR-1D-Q16STIRT IMM_M28xIMM_U28 OUTRIGHT RATE: 3.4932357259799716
USD-SOFR-1D-Q16STIRT IMM_U28xIMM_Z28 OUTRIGHT RATE: 3.521642822041087
USD-SOFR-1D-Q16STIRT IMM_Z28xIMM_H29 OUTRIGHT RATE: 3.5535380266400343
USD-SOFR-1D-Q16STIRT IMM_H29xIMM_M29 OUTRIGHT RATE: 3.5876075744640117
USD-SOFR-1D-Q16STIRT IMM_M29xIMM_U29 OUTRIGHT RATE: 3.624024286994277
USD-SOFR-1D-Q16STIRT IMM_U29xIMM_Z29 OUTRIGHT RATE: 3.6653959634774425
USD-SOFR-1

In [19]:
ts = "live"
curve = "USD-OIS-Q12xM12STIRT-SERFFX-MIX23"

curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle
queries = [
    IRSwapQuery(curve=curve, tenor="fomc_apr26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jun26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jul26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_sep26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_oct26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_dec26").resolve_query(ts, pricer_or_curve=curve_handle),
]

rl_spots = []
for q in queries:
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    print(f"{q.col_name()}: {pkg[0].rate().real}")

USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_apr26 OUTRIGHT RATE: 3.700404595891743
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jun26 OUTRIGHT RATE: 3.6835226584199483
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jul26 OUTRIGHT RATE: 3.6692755056009765
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_sep26 OUTRIGHT RATE: 3.655060599980141
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_oct26 OUTRIGHT RATE: 3.6505300051496046
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_dec26 OUTRIGHT RATE: 3.6395875776170903
